In [1]:
!pip install gradio joblib tensorflow scikit-learn

In [2]:
import numpy as np
import joblib
import tensorflow as tf
import gradio as gr
from tensorflow.keras.preprocessing import image

In [3]:
pcos_model = joblib.load("pcos_model.pkl")
pcos_scaler = joblib.load("pcos_scaler.pkl")

menstrual_model = joblib.load("menstrual_model.pkl")
menstrual_scaler = joblib.load("menstrual_scaler.pkl")

fibroid_model = tf.keras.models.load_model("fibroid_detection_model.h5")

In [4]:
PCOS_FEATURE_COUNT = 41
MENSTRUAL_FEATURE_COUNT = 6

In [5]:
def calculate_bmi(weight, height_cm):
    height_m = height_cm / 100
    return weight / (height_m ** 2)


def build_full_input(user_values, total_features):
    arr = np.zeros(total_features)
    arr[:len(user_values)] = user_values
    return arr

In [6]:
def predict_pcos(data):
    data = pcos_scaler.transform([data])
    return pcos_model.predict_proba(data)[0][1]


def predict_menstrual(data):
    data = menstrual_scaler.transform([data])
    return menstrual_model.predict_proba(data)[0][1]


def predict_fibroid(img):
    img = image.load_img(img, target_size=(224,224))
    img = image.img_to_array(img)/255.0
    img = np.expand_dims(img, axis=0)
    return fibroid_model.predict(img)[0][0]

In [7]:
def final_health_risk(pcos, menstrual, fibroid):

    score = 0.4*pcos + 0.3*menstrual + 0.3*fibroid

    if score < 0.3:
        level = "🟢 Low Risk"
    elif score < 0.7:
        level = "🟡 Moderate Risk"
    else:
        level = "🔴 High Risk"

    return score, level

In [8]:
def predict_all(
    follicle_l, follicle_r, amh, cycle_length_pcos, weight,
    cycle_length_mens, period_duration, age, height, stress,
    fibroid_img
):

    bmi = calculate_bmi(weight, height)

    pcos_user = [follicle_l, follicle_r, amh, cycle_length_pcos, weight]
    mens_user = [cycle_length_mens, period_duration, age, bmi, stress]

    pcos_full = build_full_input(pcos_user, PCOS_FEATURE_COUNT)
    mens_full = build_full_input(mens_user, MENSTRUAL_FEATURE_COUNT)

    pcos_prob = predict_pcos(pcos_full)
    mens_prob = predict_menstrual(mens_full)
    fibroid_prob = predict_fibroid(fibroid_img)

    score, level = final_health_risk(pcos_prob, mens_prob, fibroid_prob)

    report = f"""
🧾 WOMEN’S REPRODUCTIVE HEALTH AI REPORT

PCOS Risk: {pcos_prob:.2f}
Menstrual Irregularity Risk: {mens_prob:.2f}
Fibroid Risk: {fibroid_prob:.2f}

Overall Health Score: {score:.2f}

Final Assessment: {level}
"""

    return report

In [9]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🏥 ReproShield: AI for Early Detection of Female Health Disorders")

    gr.Markdown("## 🧬 PCOS Clinical Parameters")
    follicle_l = gr.Number(label="Follicle No. (Left Ovary)")
    follicle_r = gr.Number(label="Follicle No. (Right Ovary)")
    amh = gr.Number(label="AMH (ng/mL)")
    cycle_length_pcos = gr.Number(label="Cycle Length (days)")
    weight = gr.Number(label="Weight (kg)")

    gr.Markdown("## 🌸 Menstrual Health & Lifestyle")
    cycle_length_mens = gr.Number(label="Cycle Length")
    period_duration = gr.Number(label="Period Duration (days)")
    age = gr.Number(label="Age")
    height = gr.Number(label="Height (cm)")
    stress = gr.Dropdown([1,2,3,4,5], label="Stress Level (1=Low, 5=High)")

    gr.Markdown("## 🧫 Ultrasound Scan")
    fibroid_img = gr.Image(type="filepath", label="Upload Pelvic Ultrasound")

    btn = gr.Button("Generate AI Health Report")

    output = gr.Textbox(label="📋 Medical Report", lines=15)

    btn.click(
        predict_all,
        inputs=[
            follicle_l, follicle_r, amh, cycle_length_pcos, weight,
            cycle_length_mens, period_duration, age, height, stress,
            fibroid_img
        ],
        outputs=output
    )

demo.launch(share=True)

/tmp/ipykernel_427/968840406.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a8f78dc40792b9453.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install gradio joblib tensorflow scikit-learn

In [ ]:
import numpy as np
import joblib
import tensorflow as tf
import gradio as gr
from tensorflow.keras.preprocessing import image

In [ ]:
pcos_model = joblib.load("pcos_model.pkl")
pcos_scaler = joblib.load("pcos_scaler.pkl")

menstrual_model = joblib.load("menstrual_model.pkl")
menstrual_scaler = joblib.load("menstrual_scaler.pkl")

fibroid_model = tf.keras.models.load_model("fibroid_detection_model.h5")

In [ ]:
PCOS_FEATURE_COUNT = 41
MENSTRUAL_FEATURE_COUNT = 5   # ✅ corrected

In [ ]:
def calculate_bmi(weight, height_cm):
    height_m = height_cm / 100
    return weight / (height_m ** 2)


def build_full_input(user_values, total_features):
    arr = np.zeros(total_features)
    arr[:len(user_values)] = user_values
    return arr

In [ ]:
def predict_pcos(data):
    data = pcos_scaler.transform([data])
    return pcos_model.predict_proba(data)[0][1]


def predict_menstrual(data):
    data = menstrual_scaler.transform([data])
    return menstrual_model.predict_proba(data)[0][1]


def predict_fibroid(img):

    if img is None:
        return 0.0   # no image uploaded

    img = image.load_img(img, target_size=(224,224))
    img = image.img_to_array(img)/255.0
    img = np.expand_dims(img, axis=0)

    pred = fibroid_model.predict(img)[0][0]
    return float(pred)

In [ ]:
def final_health_risk(pcos, menstrual, fibroid):

    score = 0.4*pcos + 0.3*menstrual + 0.3*fibroid

    if score < 0.3:
        level = "🟢 Low Risk"
    elif score < 0.7:
        level = "🟡 Moderate Risk"
    else:
        level = "🔴 High Risk"

    return score, level

In [ ]:
def predict_all(
    follicle_l, follicle_r, amh, cycle_length_pcos, weight,
    cycle_length_mens, period_duration, age, height, stress,
    fibroid_img
):

    # ✅ Check empty fields
    if None in [follicle_l, follicle_r, amh, cycle_length_pcos, weight,
                cycle_length_mens, period_duration, age, height, stress]:
        return "⚠️ Please fill all the fields before generating the report."

    bmi = calculate_bmi(weight, height)

    # Selected important features
    pcos_user = [follicle_l, follicle_r, amh, cycle_length_pcos, weight]
    mens_user = [cycle_length_mens, period_duration, age, bmi, stress]

    # Build full feature arrays
    pcos_full = build_full_input(pcos_user, PCOS_FEATURE_COUNT)
    mens_full = build_full_input(mens_user, MENSTRUAL_FEATURE_COUNT)

    # Predictions
    pcos_prob = predict_pcos(pcos_full)
    mens_prob = predict_menstrual(mens_full)
    fibroid_prob = predict_fibroid(fibroid_img)

    score, level = final_health_risk(pcos_prob, mens_prob, fibroid_prob)

    report = f"""
🧾 WOMEN’S REPRODUCTIVE HEALTH AI REPORT

PCOS Risk: {pcos_prob:.2f}
Menstrual Irregularity Risk: {mens_prob:.2f}
Fibroid Risk: {fibroid_prob:.2f}

Overall Health Score: {score:.2f}

Final Assessment: {level}
"""

    return report

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🏥 ReproShield: AI for Early Detection of Female Health Disorders")

    gr.Markdown("## 🧬 PCOS Clinical Parameters")
    follicle_l = gr.Number(label="Follicle Count – Left Ovary")
    follicle_r = gr.Number(label="Follicle Count – Right Ovary")
    amh = gr.Number(label="Anti-Müllerian Hormone (AMH) – ng/mL")
    cycle_length_pcos = gr.Number(label="Cycle Length (days)")
    weight = gr.Number(label="Weight (kg)")

    gr.Markdown("## 🌸 Menstrual Health & Lifestyle")
    cycle_length_mens = gr.Number(label="Cycle Length")
    period_duration = gr.Number(label="Period Duration (days)")
    age = gr.Number(label="Age")
    height = gr.Number(label="Height (cm)")
    stress = gr.Dropdown([1,2,3,4,5], label="Stress Level (1=Low, 5=High)")

    gr.Markdown("## 🧫 Fibroid Detection (Ultrasound)")
    fibroid_img = gr.Image(type="filepath", label="Upload Pelvic Ultrasound (Optional)")

    btn = gr.Button("Generate AI Health Report")

    output = gr.Textbox(label="📋 Medical Report", lines=15)

    btn.click(
        predict_all,
        inputs=[
            follicle_l, follicle_r, amh, cycle_length_pcos, weight,
            cycle_length_mens, period_duration, age, height, stress,
            fibroid_img
        ],
        outputs=output
    )

demo.launch(share=True)

/tmp/ipython-input-305/3808149973.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://157e9ee7ab2d05b088.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ================= INSTALL =================
!pip install gradio joblib tensorflow scikit-learn -q

# ================= IMPORTS =================
import numpy as np
import joblib
import tensorflow as tf
import gradio as gr
from tensorflow.keras.preprocessing import image

# ================= LOAD MODELS =================
pcos_model = joblib.load("pcos_model.pkl")
pcos_scaler = joblib.load("pcos_scaler.pkl")

menstrual_model = joblib.load("menstrual_model.pkl")
menstrual_scaler = joblib.load("menstrual_scaler.pkl")

fibroid_model = tf.keras.models.load_model("fibroid_detection_model.h5")

# ================= AUTO FEATURE COUNT =================
PCOS_FEATURE_COUNT = pcos_scaler.n_features_in_
MENSTRUAL_FEATURE_COUNT = menstrual_scaler.n_features_in_

print("✅ PCOS features:", PCOS_FEATURE_COUNT)
print("✅ Menstrual features:", MENSTRUAL_FEATURE_COUNT)

# ================= HELPERS =================
def calculate_bmi(weight, height_cm):
    height_m = height_cm / 100
    return weight / (height_m ** 2)

def build_full_input(user_values, total_features):
    arr = np.zeros(total_features)
    arr[:len(user_values)] = user_values
    return arr

# ================= PREDICTION FUNCTIONS =================
def predict_pcos(data):
    data = pcos_scaler.transform([data])
    return pcos_model.predict_proba(data)[0][1]

def predict_menstrual(data):
    data = menstrual_scaler.transform([data])
    return menstrual_model.predict_proba(data)[0][1]

def predict_fibroid(img):

    if img is None:
        return 0.0

    img = image.load_img(img, target_size=(224,224))
    img = image.img_to_array(img)/255.0
    img = np.expand_dims(img, axis=0)

    pred = fibroid_model.predict(img)[0][0]
    return float(pred)

# ================= MAIN FUNCTION =================
def predict_all(
    follicle_l, follicle_r, amh, cycle_length_pcos, weight,
    cycle_length_mens, period_duration, age, height, stress,
    fibroid_img
):

    try:

        values = [follicle_l, follicle_r, amh, cycle_length_pcos, weight,
                  cycle_length_mens, period_duration, age, height, stress]

        if any(v is None for v in values):
            return "⚠️ Please fill all fields before generating report."

        values = list(map(float, values))

        bmi = calculate_bmi(values[4], values[8])

        # selected important features
        pcos_user = values[:5]
        mens_user = [values[5], values[6], values[7], bmi, values[9]]

        pcos_full = build_full_input(pcos_user, PCOS_FEATURE_COUNT)
        mens_full = build_full_input(mens_user, MENSTRUAL_FEATURE_COUNT)

        pcos_prob = predict_pcos(pcos_full)
        mens_prob = predict_menstrual(mens_full)
        fibroid_prob = predict_fibroid(fibroid_img)

        score = 0.4*pcos_prob + 0.3*mens_prob + 0.3*fibroid_prob

        if score < 0.3:
            level = "🟢 Low Risk"
        elif score < 0.7:
            level = "🟡 Moderate Risk"
        else:
            level = "🔴 High Risk"

        return f"""
🧾 WOMEN’S REPRODUCTIVE HEALTH AI REPORT

PCOS Risk: {pcos_prob:.2f}
Menstrual Irregularity Risk: {mens_prob:.2f}
Fibroid Risk: {fibroid_prob:.2f}

Overall Health Score: {score:.2f}

Final Assessment: {level}
"""

    except Exception as e:
        return f"❌ ERROR:\n{str(e)}"

# ================= GRADIO UI =================
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🏥 ReproShield: AI for Early Detection of Female Health Disorders")

    gr.Markdown("## 🧬 PCOS Clinical Parameters")
    follicle_l = gr.Number(label="Follicle Count – Left Ovary")
    follicle_r = gr.Number(label="Follicle Count – Right Ovary")
    amh = gr.Number(label="Anti-Müllerian Hormone (AMH) – ng/mL")
    cycle_length_pcos = gr.Number(label="Cycle Length (days)")
    weight = gr.Number(label="Weight (kg)")

    gr.Markdown("## 🌸 Menstrual Health & Lifestyle")
    cycle_length_mens = gr.Number(label="Cycle Length")
    period_duration = gr.Number(label="Period Duration (days)")
    age = gr.Number(label="Age")
    height = gr.Number(label="Height (cm)")
    stress = gr.Dropdown([1,2,3,4,5], label="Stress Level (1=Low → 5=High)")

    gr.Markdown("## 🧫 Fibroid Detection (Ultrasound)")
    fibroid_img = gr.Image(type="filepath", label="Upload Pelvic Ultrasound (Optional)")

    btn = gr.Button("Generate AI Health Report")

    output = gr.Textbox(label="📋 Medical Report", lines=15)

    btn.click(
        predict_all,
        inputs=[
            follicle_l, follicle_r, amh, cycle_length_pcos, weight,
            cycle_length_mens, period_duration, age, height, stress,
            fibroid_img
        ],
        outputs=output
    )

demo.launch(share=True, debug=True)

/tmp/ipython-input-305/2389557108.py:113: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


✅ PCOS features: 4
✅ Menstrual features: 9
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7abcbf01bf18fd1401.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1e94472389c0289bcf.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://157e9ee7ab2d05b088.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://7abcbf01bf18fd1401.gradio.live


In [ ]:
# ================= INSTALL =================
!pip install gradio joblib tensorflow scikit-learn -q

# ================= IMPORTS =================
import numpy as np
import joblib
import tensorflow as tf
import gradio as gr
from tensorflow.keras.preprocessing import image

# ================= LOAD MODELS =================
pcos_model = joblib.load("pcos_model.pkl")
pcos_scaler = joblib.load("pcos_scaler.pkl")

menstrual_model = joblib.load("menstrual_model.pkl")
menstrual_scaler = joblib.load("menstrual_scaler.pkl")

fibroid_model = tf.keras.models.load_model("fibroid_detection_model.h5")

PCOS_FEATURE_COUNT = pcos_scaler.n_features_in_
MENSTRUAL_FEATURE_COUNT = menstrual_scaler.n_features_in_

print("PCOS features:", PCOS_FEATURE_COUNT)
print("Menstrual features:", MENSTRUAL_FEATURE_COUNT)

# ================= HELPERS =================
def calculate_bmi(weight, height_cm):
    return weight / ((height_cm/100) ** 2)

def pad_input(values, total):
    arr = np.zeros(total)
    arr[:len(values)] = values
    return arr

# ================= PREDICT =================
def predict_all(f_l, f_r, amh, cycle_pcos, weight,
                cycle_m, period_dur, age, height, stress, fibroid_img):

    try:

        if None in [f_l,f_r,amh,cycle_pcos,weight,cycle_m,period_dur,age,height,stress]:
            return "⚠️ Fill all fields"

        bmi = calculate_bmi(weight, height)

        # ✅ PCOS → USE ONLY 4 FEATURES (your model expects 4)
        pcos_values = [f_l, f_r, amh, cycle_pcos][:PCOS_FEATURE_COUNT]
        pcos_input = pcos_scaler.transform([pcos_values])
        pcos_prob = pcos_model.predict_proba(pcos_input)[0][1]

        # ✅ MENSTRUAL → PAD TO 9 FEATURES
        mens_values = [cycle_m, period_dur, age, bmi, stress]
        mens_input = pad_input(mens_values, MENSTRUAL_FEATURE_COUNT)
        mens_input = menstrual_scaler.transform([mens_input])
        mens_prob = menstrual_model.predict_proba(mens_input)[0][1]

        # ✅ FIBROID
        if fibroid_img is None:
            fibroid_prob = 0
        else:
            img = image.load_img(fibroid_img, target_size=(224,224))
            img = image.img_to_array(img)/255.0
            img = np.expand_dims(img, axis=0)
            fibroid_prob = float(fibroid_model.predict(img)[0][0])

        # ✅ FINAL SCORE
        score = 0.4*pcos_prob + 0.3*mens_prob + 0.3*fibroid_prob

        if score < 0.3:
            level = "🟢 Low Risk"
        elif score < 0.7:
            level = "🟡 Moderate Risk"
        else:
            level = "🔴 High Risk"

        return f"""
🧾 AI REPRODUCTIVE HEALTH REPORT

PCOS Risk: {pcos_prob:.2f}
Menstrual Risk: {mens_prob:.2f}
Fibroid Risk: {fibroid_prob:.2f}

Overall Score: {score:.2f}

Final Assessment: {level}
"""

    except Exception as e:
        return str(e)

# ================= UI =================
with gr.Blocks() as demo:

    gr.Markdown("# 🏥 ReproShield: AI for Early Detection of Female Health Disorders")

    gr.Markdown("## PCOS Clinical Parameters")
    f_l = gr.Number(label="Follicle Count – Left Ovary")
    f_r = gr.Number(label="Follicle Count – Right Ovary")
    amh = gr.Number(label="AMH (ng/mL)")
    cycle_pcos = gr.Number(label="Cycle Length (days)")
    weight = gr.Number(label="Weight (kg)")

    gr.Markdown("## Menstrual Health")
    cycle_m = gr.Number(label="Cycle Length")
    period_dur = gr.Number(label="Period Duration")
    age = gr.Number(label="Age")
    height = gr.Number(label="Height (cm)")
    stress = gr.Dropdown([1,2,3,4,5], label="Stress Level")

    gr.Markdown("## Fibroid Ultrasound")
    fibroid_img = gr.Image(type="filepath")

    btn = gr.Button("Generate AI Health Report")
    output = gr.Textbox(lines=15)

    btn.click(
        predict_all,
        inputs=[f_l,f_r,amh,cycle_pcos,weight,
                cycle_m,period_dur,age,height,stress,fibroid_img],
        outputs=output
    )

demo.launch(share=True)

PCOS features: 4
Menstrual features: 9
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d580939a215aad2335.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
